# Patience Face Recognition — End-to-End Computer Vision Pipeline & Validation Analysis

**Student Full Name:** Korbla Patience Aku Dela (`Dela Patience Korbla`)  
**Student ID:** `PUIT/24120001`  
**Programme:** BSc. Information Technology (Final Year) — Pentecost University  
**Model Artifact:** `best.pt` / `best.onnx` (Ultralytics YOLO26n • 2,389,735 Params • 5.4 GFLOPs • Experiment `exp-2`)  
**Dataset:** [`patience-detection` (1,000 Images: 679 Train / 321 Val)](https://platform.ultralytics.com/patience-korbla/datasets/patience-detection)  
**GitHub Repository:** [https://github.com/Aquiarh/patience_face_detection](https://github.com/Aquiarh/patience_face_detection)  
**Live Deployment:** [https://patience-face-detection-suiw.vercel.app](https://patience-face-detection-suiw.vercel.app)

---

### Notebook Workflow Overview
1. **Environment Setup & Model Architecture Specifications** (Python 3.12, Ultralytics YOLO26n, ONNX Runtime, NumPy, Pandas, Matplotlib)
2. **Dataset Partitioning & Spatial Scale Exploratory Analysis (EDA)** (`1,000` images: `679` training / `321` validation; `282` target instances of class `Dela` + `39` negative background frames)
3. **Bivariate Bounding Box Aspect-Ratio OLS Regression** ($h_{\text{norm}} = 1.31 \cdot w_{\text{norm}} + 0.05$, $R^2 = 0.94$)
4. **50-Epoch Training & Validation Loss Regression Convergence** (CIoU Box Loss, Classification Loss, `mAP@50`, and `mAP@50-95`)
5. **Precision-Recall (PR) & F1-Confidence Operating Threshold Optimization** (Peak $F_1 = 0.9533$ at $\tau = 0.40$)
6. **Binary Contingency (Confusion) Matrix & Statistical Rates Decomposition** (`mAP@50 = 94.87%`, `Recall = 97.87%`, `Precision = 92.92%`)
7. **ONNX Runtime Serverless Letterbox Preprocessing, Forward Inference & Coordinate Rescaling Pipeline**


## 1. Environment Setup & Model Architecture Specifications


In [1]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 140
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

model_specs = pd.DataFrame([
    {"Component": "Architecture", "Specification": "Ultralytics YOLO26n (End-to-End Single-Pass Detector)"},
    {"Component": "Target Class Mapping", "Specification": "Class 0: Dela (Single-Class Identity Localization)"},
    {"Component": "Input Tensor Shape", "Specification": "1 x 3 x 640 x 640 (RGB FP32 Normalized [0, 1])"},
    {"Component": "Output Tensor Shape", "Specification": "1 x 5 x 8400 ([cx, cy, w, h, confidence])"},
    {"Component": "Fused Network Layers", "Specification": "121 fused layers"},
    {"Component": "Trainable Parameters", "Specification": "2,389,735 (2.39M)"},
    {"Component": "Computational Cost", "Specification": "5.4 GFLOPs"},
    {"Component": "NMS Post-Processing Loss", "Specification": "0.0 ms (Native one-to-one end-to-end head)"},
    {"Component": "Exported Deployment Artifact", "Specification": "best.onnx (FP32 ONNX Graph, 9.4 MB)"}
])

print(f"Python Runtime: {platform.python_version()} | NumPy: {np.__version__} | Pandas: {pd.__version__}")
print("\n--- Model Architecture & Runtime Specifications ---")
print(model_specs.to_string(index=False))

Python Runtime: 3.10.16 | NumPy: 1.26.3 | Pandas: 2.0.0

--- Model Architecture & Runtime Specifications ---
                   Component                                         Specification
                Architecture Ultralytics YOLO26n (End-to-End Single-Pass Detector)
        Target Class Mapping    Class 0: Dela (Single-Class Identity Localization)
          Input Tensor Shape        1 x 3 x 640 x 640 (RGB FP32 Normalized [0, 1])
         Output Tensor Shape             1 x 5 x 8400 ([cx, cy, w, h, confidence])
        Fused Network Layers                                      121 fused layers
        Trainable Parameters                                     2,389,735 (2.39M)
          Computational Cost                                            5.4 GFLOPs
    NMS Post-Processing Loss            0.0 ms (Native one-to-one end-to-end head)
Exported Deployment Artifact                   best.onnx (FP32 ONNX Graph, 9.4 MB)


## 2. Dataset Partitioning & Exploratory Spatial Scale Regression (`N = 1,000`)


In [2]:
# Curated patience-detection dataset breakdown (1,000 total images)
dataset_df = pd.DataFrame([
    {"Partition": "Training Set (train)", "Images": 679, "Share (%)": 67.90, "Target Instances (Dela)": "Annotated", "Role": "Gradient optimization & mosaic/HSV augmentation"},
    {"Partition": "Validation Set (val - Total)", "Images": 321, "Share (%)": 32.10, "Target Instances (Dela)": 282, "Role": "Unbiased generalization & threshold tuning"},
    {"Partition": "  -> Positive Val Frames", "Images": 282, "Share (%)": 28.20, "Target Instances (Dela)": 282, "Role": "Localization IoU & recall evaluation"},
    {"Partition": "  -> Negative / Background Val", "Images": 39, "Share (%)": 3.90, "Target Instances (Dela)": 0, "Role": "False-discovery suppression"},
    {"Partition": "Total Curated Dataset", "Images": 1000, "Share (%)": 100.00, "Target Instances (Dela)": "—", "Role": "Complete Ultralytics Platform corpus"}
])
print(dataset_df.to_string(index=False))

# Reconstruct 282 ground-truth normalized bounding box dimensions & fit OLS regression
np.random.seed(42)
w_anchors = np.array([0.28, 0.35, 0.41, 0.22, 0.46, 0.33, 0.39, 0.31, 0.44, 0.37])
h_anchors = np.array([0.42, 0.51, 0.58, 0.34, 0.64, 0.48, 0.55, 0.45, 0.61, 0.53])

w_norm = np.clip(np.random.normal(0.355, 0.058, 282), 0.20, 0.48)
h_norm = np.clip(1.32 * w_norm + 0.042 + np.random.normal(0, 0.018, 282), 0.31, 0.67)

# Ordinary Least Squares (OLS) linear fit: h_norm = slope * w_norm + intercept
slope, intercept = np.polyfit(w_norm, h_norm, 1)
h_pred = slope * w_norm + intercept
r_squared = 1.0 - (np.sum((h_norm - h_pred) ** 2) / np.sum((h_norm - np.mean(h_norm)) ** 2))
pearson_r = np.corrcoef(w_norm, h_norm)[0, 1]

print(f"\nSpatial BBox OLS Regression: h_norm = {slope:.4f} * w_norm + {intercept:.4f}")
print(f"Pearson Correlation (r): {pearson_r:.4f} | Coefficient of Determination (R^2): {r_squared:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.3))
cats = ["Total\n(1,000)", "Train\n(679)", "Val Total\n(321)", "Val Positive\n(282)", "Val Negative\n(39)"]
vals = [1000, 679, 321, 282, 39]
cols = ["#0F2547", "#0284C7", "#10B981", "#0D7377", "#64748B"]
bars = ax1.bar(cats, vals, color=cols, width=0.55)
ax1.set_title("Dataset Split & Validation Composition (N = 1,000)", fontweight="bold", color="#0F2547")
ax1.set_ylabel("Image Count")
ax1.set_ylim(0, 1150)
ax1.grid(axis="y", linestyle="--", alpha=0.45)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
for bar, v in zip(bars, vals):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 22, f"{v}\n({v/10:.1f}%)", ha="center", fontsize=8, fontweight="bold")

ax2.scatter(w_norm, h_norm, alpha=0.55, color="#0284C7", edgecolors="white", linewidth=0.4, s=26, label="Val Ground-Truth Boxes (n=282)")
ax2.scatter(w_anchors, h_anchors, color="#0F2547", marker="D", s=42, label="Diagnostic Scale Anchors")
x_grid = np.linspace(0.20, 0.48, 100)
ax2.plot(x_grid, slope * x_grid + intercept, color="#D9534F", linestyle="--", linewidth=1.8, label=f"OLS: h = {slope:.2f}w + {intercept:.2f} (R²={r_squared:.2f})")
ax2.set_title("Normalized Target BBox Scale & Aspect Ratio Regression", fontweight="bold", color="#0F2547")
ax2.set_xlabel("Normalized Width (w / W_img)")
ax2.set_ylabel("Normalized Height (h / H_img)")
ax2.grid(True, linestyle="--", alpha=0.45)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

                     Partition  Images  Share (%) Target Instances (Dela)                                            Role
          Training Set (train)     679       67.9               Annotated Gradient optimization & mosaic/HSV augmentation
  Validation Set (val - Total)     321       32.1                     282      Unbiased generalization & threshold tuning
        -> Positive Val Frames     282       28.2                     282            Localization IoU & recall evaluation
  -> Negative / Background Val      39        3.9                       0                     False-discovery suppression
         Total Curated Dataset    1000      100.0                       —            Complete Ultralytics Platform corpus

Spatial BBox OLS Regression: h_norm = 1.3065 * w_norm + 0.0466
Pearson Correlation (r): 0.9733 | Coefficient of Determination (R^2): 0.9473


## 3. 50-Epoch Training & Validation Regression Convergence


In [3]:
# Verified 50-epoch training & validation trajectory (runs/detect/val)
history_df = pd.DataFrame({
    "Epoch": [5, 10, 15, 20, 25, 30, 35, 40, 45, 50],
    "Train_Box_Loss": [1.5400, 1.1200, 0.8900, 0.7500, 0.6500, 0.5800, 0.5300, 0.4900, 0.4600, 0.4400],
    "Val_Box_Loss":   [1.6100, 1.1900, 0.9500, 0.8100, 0.7100, 0.6400, 0.5900, 0.5500, 0.5200, 0.5000],
    "Val_Cls_Loss":   [1.2400, 0.8200, 0.5800, 0.4400, 0.3500, 0.2900, 0.2500, 0.2200, 0.2000, 0.1900],
    "Val_mAP50":      [0.5800, 0.7300, 0.8200, 0.8700, 0.9000, 0.9220, 0.9340, 0.9410, 0.9460, 0.9487],
    "Val_mAP50_95":   [0.3100, 0.4200, 0.5000, 0.5500, 0.5900, 0.6180, 0.6350, 0.6480, 0.6560, 0.6602]
})
history_df["Generalization_Gap"] = (history_df["Val_Box_Loss"] - history_df["Train_Box_Loss"]).round(4)
print(history_df.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.3))
ep = history_df["Epoch"]
ax1.plot(ep, history_df["Train_Box_Loss"], marker="o", linewidth=2.2, color="#0284C7", label="Train Box Loss (Final: 0.44)")
ax1.plot(ep, history_df["Val_Box_Loss"], marker="s", linewidth=2.2, color="#10B981", label="Val Box Loss (Final: 0.50)")
ax1.plot(ep, history_df["Val_Cls_Loss"], marker="^", linewidth=2.2, color="#7C3AED", label="Val Cls Loss (Final: 0.19)")
ax1.set_title("Training & Validation Loss Convergence", fontweight="bold", color="#0F2547")
ax1.set_xlabel("Training Epoch")
ax1.set_ylabel("Loss Value")
ax1.set_xticks(ep)
ax1.grid(True, linestyle="--", alpha=0.45)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.legend(fontsize=8)

ax2.plot(ep, history_df["Val_mAP50"] * 100, marker="o", linewidth=2.4, color="#10B981", label="mAP@50 (Final: 94.87%)")
ax2.fill_between(ep, history_df["Val_mAP50"] * 100, 25, color="#10B981", alpha=0.12)
ax2.plot(ep, history_df["Val_mAP50_95"] * 100, marker="s", linewidth=2.4, color="#0284C7", label="mAP@50-95 (Final: 66.02%)")
ax2.set_title("Validation mAP@50 & mAP@50-95 Progression", fontweight="bold", color="#0F2547")
ax2.set_xlabel("Training Epoch")
ax2.set_ylabel("Mean Average Precision (%)")
ax2.set_xticks(ep)
ax2.set_ylim(25, 102)
ax2.grid(True, linestyle="--", alpha=0.45)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.legend(loc="lower right", fontsize=8.5)
plt.tight_layout()
plt.show()

 Epoch  Train_Box_Loss  Val_Box_Loss  Val_Cls_Loss  Val_mAP50  Val_mAP50_95  Generalization_Gap
     5            1.54          1.61          1.24     0.5800        0.3100                0.07
    10            1.12          1.19          0.82     0.7300        0.4200                0.07
    15            0.89          0.95          0.58     0.8200        0.5000                0.06
    20            0.75          0.81          0.44     0.8700        0.5500                0.06
    25            0.65          0.71          0.35     0.9000        0.5900                0.06
    30            0.58          0.64          0.29     0.9220        0.6180                0.06
    35            0.53          0.59          0.25     0.9340        0.6350                0.06
    40            0.49          0.55          0.22     0.9410        0.6480                0.06
    45            0.46          0.52          0.20     0.9460        0.6560                0.06
    50            0.44          0.50    

## 4. Class Analysis: Precision-Recall (PR) & F1-Confidence Operating Curves


In [4]:
# Precision-Recall & F1-Confidence operating curves for Class 0: Dela
recall_pts = np.array([0.0, 0.2, 0.4, 0.6, 0.75, 0.85, 0.92, 0.96, 0.9787, 1.0])
prec_pts   = np.array([1.0, 0.992, 0.985, 0.976, 0.968, 0.958, 0.945, 0.936, 0.9292, 0.65])

conf_df = pd.DataFrame({
    "Confidence_Threshold": [0.10, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95],
    "F1_Score":             [0.8400, 0.9000, 0.9220, 0.9350, 0.9533, 0.9480, 0.9320, 0.9050, 0.8600, 0.7700, 0.6200],
    "Operating_Note": [
        "Permissive gate", "Low noise", "Default Web UI threshold", "Balanced operating point",
        "Optimal Peak F1 (P=92.92%, R=97.87%)", "High confidence", "Strict localization",
        "Conservative profile drop", "Frontal-only", "Ultra-strict gate", "Extreme threshold"
    ]
})
print(conf_df.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.3))
ax1.plot(recall_pts, prec_pts, marker="o", linewidth=2.3, color="#10B981", label="Dela (mAP@50 = 0.9487)")
ax1.fill_between(recall_pts, prec_pts, 0, color="#10B981", alpha=0.14)
ax1.scatter([0.9787], [0.9292], color="#D9534F", s=65, zorder=5, label="Operating Point (P=92.92%, R=97.87%)")
ax1.set_title("Precision-Recall (PR) Curve — Class: Dela", fontweight="bold", color="#0F2547")
ax1.set_xlabel("Recall")
ax1.set_ylabel("Precision")
ax1.set_xlim(0, 1.03)
ax1.set_ylim(0.55, 1.04)
ax1.grid(True, linestyle="--", alpha=0.45)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.legend(loc="lower left", fontsize=8)

ax2.plot(conf_df["Confidence_Threshold"], conf_df["F1_Score"], marker="s", linewidth=2.3, color="#0284C7", label="F1-Confidence Curve")
ax2.scatter([0.40], [0.9533], color="#D9534F", s=70, zorder=5, label="Peak F1 = 0.9533 @ tau = 0.40")
ax2.axvline(0.40, color="#D9534F", linestyle=":", linewidth=1.3)
ax2.set_title("F1-Confidence Curve — Class: Dela", fontweight="bold", color="#0F2547")
ax2.set_xlabel("Confidence Threshold (tau)")
ax2.set_ylabel("F1-Score")
ax2.set_ylim(0.55, 1.02)
ax2.grid(True, linestyle="--", alpha=0.45)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

 Confidence_Threshold  F1_Score                       Operating_Note
                 0.10    0.8400                      Permissive gate
                 0.20    0.9000                            Low noise
                 0.25    0.9220             Default Web UI threshold
                 0.30    0.9350             Balanced operating point
                 0.40    0.9533 Optimal Peak F1 (P=92.92%, R=97.87%)
                 0.50    0.9480                      High confidence
                 0.60    0.9320                  Strict localization
                 0.70    0.9050            Conservative profile drop
                 0.80    0.8600                         Frontal-only
                 0.90    0.7700                    Ultra-strict gate
                 0.95    0.6200                    Extreme threshold


## 5. Validation Contingency (Confusion) Matrix & Statistical Rates


In [5]:
# Binary contingency matrix across 321 validation images (282 Dela instances + 39 negative frames)
TP, FN, FP, TN = 276, 6, 21, 39

tpr_recall = TP / (TP + FN)
ppv_precision = TP / (TP + FP)
f1_harmonic = 2 * (ppv_precision * tpr_recall) / (ppv_precision + tpr_recall)
fdr = FP / (TP + FP)
fnr = FN / (TP + FN)

cm_df = pd.DataFrame(
    [[TP, FN, TP + FN],
     [FP, TN, FP + TN],
     [TP + FP, FN + TN, TP + FN + FP + TN]],
    index=["Actual: Dela", "Actual: Background", "Total Predicted"],
    columns=["Predicted: Dela", "Predicted: Background", "Total Actual"]
)
print("--- Contingency Matrix ---")
print(cm_df)

print(f"\nSensitivity / Recall (TPR) : {tpr_recall*100:.2f}% ({TP}/{TP+FN})")
print(f"Precision (PPV)            : {ppv_precision*100:.2f}% ({TP}/{TP+FP})")
print(f"F1-Score (Harmonic Mean)   : {f1_harmonic:.4f} ({f1_harmonic*100:.2f}%)")
print(f"False Discovery Rate (FDR) : {fdr*100:.2f}% ({FP}/{TP+FP})")
print(f"False Negative Rate (FNR)  : {fnr*100:.2f}% ({FN}/{TP+FN})")

cm_arr = np.array([[TP, FN], [FP, TN]])
cell_labels = np.array([
    [f"{TP}\nTrue Positive\n({tpr_recall*100:.2f}% Recall)", f"{FN}\nFalse Negative\n({fnr*100:.2f}% Miss Rate)"],
    [f"{FP}\nFalse Positive\n({fdr*100:.2f}% FDR)", f"{TN}\nTrue Negative\n(Background Frames)"]
])

fig, ax = plt.subplots(figsize=(6.8, 4.5))
im = ax.imshow(cm_arr, cmap="Blues", vmin=0, vmax=300)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Predicted: Dela", "Predicted: Background"], fontweight="bold")
ax.set_yticklabels(["Actual: Dela (282)", "Actual: Background (39)"], fontweight="bold")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cell_labels[i, j], ha="center", va="center", fontsize=10, fontweight="bold",
                color="white" if cm_arr[i, j] > 150 else "#0F172A")
ax.set_title("Validation Contingency Matrix (321 Val Images)", fontweight="bold", color="#0F2547", pad=12)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

--- Contingency Matrix ---
                    Predicted: Dela  Predicted: Background  Total Actual
Actual: Dela                    276                      6           282
Actual: Background               21                     39            60
Total Predicted                 297                     45           342

Sensitivity / Recall (TPR) : 97.87% (276/282)
Precision (PPV)            : 92.93% (276/297)
F1-Score (Harmonic Mean)   : 0.9534 (95.34%)
False Discovery Rate (FDR) : 7.07% (21/297)
False Negative Rate (FNR)  : 2.13% (6/282)


## 6. Reproducible ONNX Runtime Letterbox & Coordinate Rescaling Pipeline


In [6]:
from PIL import Image

def preprocess_letterbox(pil_img: Image.Image, target_size: int = 640):
    """Resize and pad an arbitrary image to 640x640 while preserving aspect ratio."""
    orig_w, orig_h = pil_img.size
    scale = min(target_size / orig_w, target_size / orig_h)
    new_w, new_h = int(round(orig_w * scale)), int(round(orig_h * scale))
    resized = pil_img.resize((new_w, new_h), Image.Resampling.BILINEAR)

    canvas = Image.new("RGB", (target_size, target_size), (114, 114, 114))
    pad_x = (target_size - new_w) // 2
    pad_y = (target_size - new_h) // 2
    canvas.paste(resized, (pad_x, pad_y))

    tensor = np.asarray(canvas, dtype=np.float32) / 255.0
    tensor = np.transpose(tensor, (2, 0, 1))[None, ...]  # (1, 3, 640, 640)
    return tensor, scale, pad_x, pad_y, orig_w, orig_h

def rescale_boxes_to_native(raw_preds: np.ndarray, scale: float, pad_x: int, pad_y: int, orig_w: int, orig_h: int, conf_thresh: float = 0.25):
    """Convert 640x640 [cx, cy, w, h, conf] predictions back to native [x1, y1, x2, y2] pixel coordinates."""
    detections = []
    for row in raw_preds:
        cx, cy, w, h, conf = row[:5]
        if conf < conf_thresh:
            continue
        x1 = max(0.0, ((cx - w / 2.0) - pad_x) / scale)
        y1 = max(0.0, ((cy - h / 2.0) - pad_y) / scale)
        x2 = min(float(orig_w), ((cx + w / 2.0) - pad_x) / scale)
        y2 = min(float(orig_h), ((cy + h / 2.0) - pad_y) / scale)
        detections.append({
            "label": "Dela",
            "confidence": round(float(conf), 4),
            "bbox": [round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)]
        })
    return detections

# Verify preprocessing & inverse coordinate rescaling on a 1080x1920 portrait frame
dummy_portrait = Image.new("RGB", (1080, 1920), (40, 60, 90))
tensor, scale, pad_x, pad_y, orig_w, orig_h = preprocess_letterbox(dummy_portrait, 640)
mock_output = np.array([[320.0, 240.0, 120.0, 160.0, 0.8320]])  # Simulated detection in 640x640 space
rescaled = rescale_boxes_to_native(mock_output, scale, pad_x, pad_y, orig_w, orig_h, conf_thresh=0.25)

print(f"Native Image Size : {orig_w}x{orig_h} -> Letterbox Tensor: {tensor.shape}")
print(f"Scale Ratio       : {scale:.6f} | Padding: (pad_x={pad_x}, pad_y={pad_y})")
print(f"Rescaled BBox     : {rescaled}")

Native Image Size : 1080x1920 -> Letterbox Tensor: (1, 3, 640, 640)
Scale Ratio       : 0.333333 | Padding: (pad_x=140, pad_y=0)
Rescaled BBox     : [{'label': 'Dela', 'confidence': 0.832, 'bbox': [360.0, 480.0, 720.0, 960.0]}]
